In [1]:
from utils import * 
from Bio import Entrez
Entrez.email = 'prichter@berkeley.edu'
%load_ext autoreload 
%autoreload 2 

# /groups/banfield/ggkbase/exports/pippa/pippa_contigs_project_name_descript_location.txt

In [2]:
# Initially tried comparing the library sizes to identify which samples the biotite reads belong to, but it was not really working
# (too many differences, too much ambiguity). Decided to just download and set up everything using the SRAs. 

# srp_ids = ['SRP080092', 'SRP080091', 'SRP080093', 'SRP080096', 'SRP080097', 'SRP080103', 'SRP080477', 'SRP080474', 'SRP080106', 'SRP074902', 'SRP080105', 'SRP080107', 'SRP080475', 'SRP080470', 'SRP080473', 'SRP080472', 'SRP080478', 'SRP080661', 'SRP080658', 'SRP080506', 'SRP080496', 'SRP080660', 'SRP080664']
# for srp_id in srp_ids:
#     result = Entrez.esearch(db='sra', term=srp_id)
#     result = Entrez.read(result)
#     assert len(result['IdList']) == 1, 'Only expecting one ID.'
#     result = Entrez.efetch(db='sra', id=result['IdList'][0], retmode='text').read().decode()
#     srr_id = re.search(r'SRR(\d)+', result).group(0)
#     print(srr_id)

In [3]:
project_id_map = pd.read_csv('../data/blast/ggkbase/blast_ggkbase_contig_projects.txt', sep='\t', usecols=[0, 1], names=['contig_id', 'project_id']).set_index('contig_id').project_id.dropna().to_dict()
contig_metadata_df = pd.read_csv('../data/blast/ggkbase/blast_ggkbase_contig_metadata.csv')
contig_metadata_df = contig_metadata_df[contig_metadata_df.location.str.contains('rifle_sediment')].copy()
contig_metadata_df['project_id'] = contig_metadata_df.contig_id.map(project_id_map)

In [4]:
get_year = lambda project_id : int(re.search(r'CSP(\d)', project_id).group(1)) if (re.search(r'CSP(\d)', project_id) is not None) else None
get_depth = lambda project_id : int(re.search(r'(\d+)ft', project_id).group(1)) if (re.search(r'(\d+)ft', project_id) is not None) else None
get_replicate = lambda project_id : int(re.findall(r'\d', project_id)[-1])

year_map = dict()
year_map.update({project_id:get_year(project_id) for project_id in contig_metadata_df.project_id.unique()})
year_map['16ft_4'] = 1 # Only one of these replicate and depth combinations, in CSP 1.
year_map['RifCSP_19_4_full'] = 1 # Only one of these replicate and depth combinations, in CSP 1.
year_map['19ft_2_thinned'] = 2

depth_map = dict()
depth_map.update({project_id:get_depth(project_id) for project_id in contig_metadata_df.project_id.unique()})
depth_map['RifCSP_19_1_full'] = 19
depth_map['RifCSP_19_4_full'] = 19
depth_map['RiFCSP_10_2_full'] = 10
depth_map['RifCSP_10_1_full'] = 10
depth_map['RiFCSP_13_1_full'] = 13

contig_metadata_df['depth'] = contig_metadata_df.project_id.map(depth_map)
contig_metadata_df['year'] = contig_metadata_df.project_id.map(year_map)
contig_metadata_df['replicate'] = contig_metadata_df.project_id.apply(get_replicate) 
contig_metadata_df['year'] = contig_metadata_df.year.fillna(0)
contig_metadata_df['sample_id'] = [f'sed_csp{int(row.year)}_{int(row.depth)}ft_{row.replicate}' for row in contig_metadata_df.itertuples()]


In [5]:
depth_map['Rifle Sediment CSP1 13ft rep2']
contig_metadata_df[contig_metadata_df.project_id == 'Rifle Sediment CSP1 13ft rep2']
# sample_metadata_df[sample_metadata_df.project_id == 'Rifle Sediment CSP1 13ft rep2']


,contig_id,location,coverage,project_id,depth,year,replicate,sample_id
1935,RifSed_csp1_13ft_2_scaffold_42941,rifle_sediment,4.0,Rifle Sediment CSP1 13ft rep2,13,1.0,2,sed_csp1_13ft_2


In [6]:
sample_metadata_df = pd.read_csv('hug_2015_table_1.csv')
sample_metadata_df = sample_metadata_df.dropna()
sample_metadata_df['year'] = [int(re.search(r'sed_csp(\d)', sample_id).group(1)) for sample_id in sample_metadata_df.sample_id]
sample_metadata_df['replicate'] = [int(re.search(r'sed_csp\d_\d\dft_(\d)', sample_id).group(1)) for sample_id in sample_metadata_df.sample_id]
sample_metadata_df['depth'] = sample_metadata_df.depth_or_filter_size.map({'3 m':10, '4 m':13, '5 m':16, '6 m':19})
sample_metadata_df = sample_metadata_df.merge(contig_metadata_df[['project_id', 'depth', 'year', 'replicate']].drop_duplicates('project_id'), on=['depth', 'year', 'replicate'])

In [7]:
paired_end_script = '''#!/bin/bash 

#SBATCH --job-name={sample_id}
#SBATCH --output={sample_id}.out
#SBATCH --cpus-per-task={num_threads}

cd {ncbi_dir}

mkdir -p {output_dir}
mkdir -p {tmp}
prefetch {srr_id} --max-size {max_size}

fasterq-dump {srr_id} --split-files -e {num_threads} -O {output_dir} --temp {tmp}
mv {fasterq_forward_reads_path} {forward_reads_path}
mv {fasterq_reverse_reads_path} {reverse_reads_path}

sickle pe -f {forward_reads_path} -r {reverse_reads_path} -t sanger -o {trimmed_forward_reads_path} -p {trimmed_reverse_reads_path} -s {singles_reads_path} -q {q} -l {l}
pigz -p {num_threads} {trimmed_forward_reads_path}
pigz -p {num_threads} {trimmed_reverse_reads_path}
'''



In [ ]:
forward_reads_paths, reverse_reads_paths = dict(), dict()

for row in sample_metadata_df.itertuples():
    sample_id, srr_id = row.sample_id, row.srr_id

    params = dict()
    params['srr_id'] = srr_id
    params['sample_id'] = sample_id
    params['output_dir'] = f'/groups/banfield/scratch/projects/environmental/sr/int/betazoid/rifle_sediment/{sample_id}'
    params['ncbi_dir'] = f'/groups/banfield/scratch/projects/environmental/sr/int/betazoid/ncbi' # Directory for the SRA cache thing that prefetch and fasterq-dump use. 
    # params['srr_path'] = os.path.join(params['ncbi_dir'], srr_id, f'{srr_id}.sra')
    params['srr_path'] = os.path.join(params['ncbi_dir'], srr_id)
    params['tmp'] = f'/groups/banfield/scratch/projects/environmental/sr/int/betazoid/tmp'
    params['forward_reads_path'] = os.path.join(params['output_dir'], f'{sample_id}.PE.1.fastq')
    params['reverse_reads_path'] = os.path.join(params['output_dir'], f'{sample_id}.PE.2.fastq')
    params['reads_path'] = os.path.join(params['output_dir'], f'{sample_id}.fastq')
    params['singles_reads_path'] = os.path.join(params['output_dir'], f'singles.fastq')
    params['trimmed_forward_reads_path'] = os.path.join(params['output_dir'], f'{sample_id}.trimmed.PE.1.fastq')
    params['trimmed_reverse_reads_path'] = os.path.join(params['output_dir'], f'{sample_id}.trimmed.PE.2.fastq')
    params['fasterq_forward_reads_path'] = os.path.join(params['output_dir'], f'{srr_id}_1.fastq')
    params['fasterq_reverse_reads_path'] = os.path.join(params['output_dir'], f'{srr_id}_2.fastq')
    params['fasterq_reads_path'] = os.path.join(params['output_dir'], f'{srr_id}.fastq')
    params['max_size'] = 4000000000
    params['num_threads'] = 16
    params['q'] = 20 
    params['l'] = 50

    # script.format(**params)
    forward_reads_path, reverse_reads_path = params['forward_reads_path'], params['reverse_reads_path']
    forward_reads_paths[sample_id], reverse_reads_paths[sample_id] = forward_reads_path, reverse_reads_path 
    # print(f'{row.sample_id}, {row.project_id}\t{forward_reads_path}\t{reverse_reads_path}')
    

In [9]:
cluster_df = pd.read_csv('blast_ggkbase_contigs_cluster.tsv', sep='\t', names=['rep_id', 'id'])
cluster_df = cluster_df[cluster_df['id'].isin(contig_metadata_df.contig_id.unique())] # Get the Rifle sediment contigs. 
cluster_df[cluster_df['id'] == 'RifSed_csp1_16ft_4_scaffold_2135']

contig_id = 'RifSed_csp1_16ft_4_scaffold_2135'
bin_id = contig_id.split('_scaffold')[0]

contig_cluster_id = cluster_df[cluster_df['id'] == contig_id].rep_id.iloc[0]
for row in cluster_df[cluster_df.rep_id == contig_cluster_id].itertuples():
    print(row.id)

print()
contig_metadata_df[(contig_metadata_df.depth == 16) & (contig_metadata_df.year == 1)]

16ft_4_scaffold_2133
RifSed_csp1_16ft_4_scaffold_2135
RifSed_csp1_19ft_3_scaffold_3516



,contig_id,location,coverage,project_id,depth,year,replicate,sample_id
758,RifSed_csp1_16ft_1_scaffold_214145,rifle_sediment,4.0,Rifle Sediment CSP1 16ft rep1,16,1.0,1,sed_csp1_16ft_1
772,RifSed_csp1_16ft_3_scaffold_99051,rifle_sediment,4.0,Rifle Sediment CSP1 16ft rep3,16,1.0,3,sed_csp1_16ft_3
774,16ft_4_scaffold_2133,rifle_sediment,5.0,16ft_4,16,1.0,4,sed_csp1_16ft_4
777,RifSed_csp1_16ft_4_scaffold_2135,rifle_sediment,10.0,Rifle Sediment CSP1 16ft rep4,16,1.0,4,sed_csp1_16ft_4
778,RifSed_csp1_16ft_2_scaffold_150438,rifle_sediment,3.0,Rifle Sediment CSP1 16ft rep2,16,1.0,2,sed_csp1_16ft_2
884,RifSed_csp1_16ft_3_scaffold_139099,rifle_sediment,4.0,Rifle Sediment CSP1 16ft rep3,16,1.0,3,sed_csp1_16ft_3
885,RifSed_csp1_16ft_2_scaffold_82342,rifle_sediment,4.0,Rifle Sediment CSP1 16ft rep2,16,1.0,2,sed_csp1_16ft_2
887,RifSed_csp1_16ft_1_scaffold_315404,rifle_sediment,3.0,Rifle Sediment CSP1 16ft rep1,16,1.0,1,sed_csp1_16ft_1
960,RifSed_csp1_16ft_1_scaffold_321683,rifle_sediment,3.0,Rifle Sediment CSP1 16ft rep1,16,1.0,1,sed_csp1_16ft_1
1857,RifSed_csp1_16ft_1_scaffold_192021,rifle_sediment,12.0,Rifle Sediment CSP1 16ft rep1,16,1.0,1,sed_csp1_16ft_1


In [10]:
# Want to take a look at the relationship between the Rifle contigs, especially because some of them don't seem to collapse into
# clusters despite having high-similarity regions. 
get_gc_content = lambda seq : (seq.count('G') + seq.count('C')) / len(seq)
contig_ids = contig_metadata_df.contig_id.unique()
fasta_df = FASTAFile.from_file('../data/blast/ggkbase/blast_ggkbase_contigs.fasta', filter_=lambda row : row.id in contig_ids).to_df()
fasta_df['gc_content'] = fasta_df.seq.apply(get_gc_content)
fasta_df = fasta_df[fasta_df.gc_content < 0.3].copy() # Remove the false positives, but keep the ones that are too short to curate.
FASTAFile.from_df(fasta_df).write('blast_ggkbase_contigs_rifle_sediment.fasta')


In [3]:
# ! progressiveMauve --output="blast_ggkbase_contigs_rifle_sediment.xmfa" "blast_ggkbase_contigs_rifle_sediment.fasta"
! minimap2 --cs=long -x asm5 -X "blast_ggkbase_contigs_rifle_sediment.fasta" "blast_ggkbase_contigs_rifle_sediment.fasta" > "blast_ggkbase_contigs_rifle_sediment.paf"
! paftools.js view "blast_ggkbase_contigs_rifle_sediment.paf" > "blast_ggkbase_contigs_rifle_sediment.txt"

[M::mm_idx_gen::0.004*1.37] collected minimizers
[M::mm_idx_gen::0.006*1.69] sorted minimizers
[M::main::0.006*1.68] loaded/built the index for 44 target sequence(s)
[M::mm_mapopt_update::0.006*1.66] mid_occ = 50
[M::mm_idx_stat] kmer size: 19; skip: 19; is_hpc: 0; #seq: 44
[M::mm_idx_stat::0.006*1.65] distinct minimizers: 5043 (16.38% are singletons); average occurrences: 3.784; average spacing: 10.024; total length: 191287
[M::worker_pipeline::0.089*2.46] mapped 44 sequences
[M::main] Version: 2.30-r1287
[M::main] CMD: minimap2 --cs=long -x asm5 -X blast_ggkbase_contigs_rifle_sediment.fasta blast_ggkbase_contigs_rifle_sediment.fasta
[M::main] Real time: 0.089 sec; CPU: 0.219 sec; Peak RSS: 0.117 GB


In [12]:
# single_end_script = '''#!/bin/bash 

# #SBATCH --job-name={sample_id}
# #SBATCH --output={sample_id}.out
# #SBATCH --cpus-per-task={num_threads}

# cd {ncbi_dir}

# mkdir -p {output_dir}
# mkdir -p {tmp}
# prefetch {srr_id} --max-size {max_size}

# fasterq-dump {srr_id} --split-files -e {num_threads} -O {output_dir} --temp {tmp}
# mv {fasterq_reads_path} {reads_path}

# sickle se -f {reads_path}  -t sanger -o {trimmed_reads_path} -q {q} -l {l}
# pigz -p {num_threads} {reads_path}
# '''